# Neurax Agent Fine-Tuning on Google Colab

This notebook fine-tunes Qwen 2.5-3B on the Neurax agent dataset using LoRA.

**Estimated time on T4 GPU**: ~30-60 minutes for 3 epochs

In [ ]:
# @title 1. Setup Environment
!pip install -q torch transformers datasets peft trl accelerate bitsandbytes

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from google.colab import files
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# @title 2. Upload Dataset
print("Upload your training dataset (neurax_train.jsonl):")
uploaded_train = files.upload()
train_file = list(uploaded_train.keys())[0]

print("\nUpload your validation dataset (neurax_valid.jsonl):")
uploaded_valid = files.upload()
valid_file = list(uploaded_valid.keys())[0]

print(f"\nTraining file: {train_file}")
print(f"Validation file: {valid_file}")

In [ ]:
# @title 3. Load and Prepare Dataset
def load_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return Dataset.from_list(data)

train_dataset = load_jsonl(train_file)
valid_dataset = load_jsonl(valid_file)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(valid_dataset)}")

In [ ]:
# @title 4. Load Model with 4-bit Quantization
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # @param ["Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct", "THUDM/glm-4-9b-chat"]

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Prepare for LoRA training
model = prepare_model_for_kbit_training(model)

print("Model loaded successfully!")

In [ ]:
# @title 5. Configure LoRA
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# @title 6. Format Dataset
def format_conversation(example, tokenizer):
    """Format conversation to ChatML format."""
    text = ""
    for conv in example.get("conversations", []):
        role = conv.get("from", "")
        value = conv.get("value", "")
        if role == "system":
            text += f"<|im_start|>system\n{value}<|im_end|>\n"
        elif role == "user":
            text += f"<|im_start|>user\n{value}<|im_end|>\n"
        elif role == "assistant":
            text += f"<|im_start|>assistant\n{value}<|im_end|>\n"
    return text

def formatting_func(example):
    return {"text": format_conversation(example, tokenizer)}

train_dataset = train_dataset.map(formatting_func)
valid_dataset = valid_dataset.map(formatting_func)

print("Dataset formatted!")
print(f"Sample: {train_dataset[0]['text'][:200]}...")

In [ ]:
# @title 7. Training Configuration
EPOCHS = 3  # @param {type:"integer"}
BATCH_SIZE = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
MAX_SEQ_LENGTH = 1024  # @param {type:"integer"}

training_args = SFTConfig(
    output_dir="./neurax-lora",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    push_to_hub=False,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
)

In [ ]:
# @title 8. Start Training
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    args=training_args,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# @title 9. Save Model
OUTPUT_DIR = "./neurax-lora-final"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# @title 10. Download Trained Model
import shutil
from google.colab import files

# Create zip file
shutil.make_archive("neurax-lora", 'zip', OUTPUT_DIR)

# Download
files.download("neurax-lora.zip")

print("Download started!")

In [ ]:
# @title 11. Test Inference (Optional)
def generate_response(prompt, model, tokenizer, max_new_tokens=512):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompt = """<|im_start|>system
You are Neurax AI controller. You build neural network architectures using tools.

TOOLS: set_family, add_node, connect, set_node_params, done

OUTPUT FORMAT: Return ONLY valid JSON:
{"assistant": "<reasoning>", "tool": {"name": "<tool_name>", "args": {...}}}
<|im_end|>
<|im_start|>user
Build a CNN for image classification with 10 classes<|im_end|>
<|im_start|>assistant
"""

print("Testing inference...")
response = generate_response(test_prompt, model, tokenizer)
print(response)